<a href="https://colab.research.google.com/github/Aiman-Naheed-Iqbal/ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiman-Naheed-Iqbal/ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# Import core modeling libraries for Section 1 setup
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully for modeling phase.")


#We chose a Random Forest Classifier (with Logistic Regression as a linear baseline comparator).


Libraries imported successfully for modeling phase.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, GroupKFold

# 1. Automatically locate CSV files in your workspace
csv_files = []
for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv") and not file.startswith("."):
            csv_files.append(os.path.join(root, file))

print("Found CSV files:", csv_files)

if not csv_files:
    raise FileNotFoundError("No CSV file found in the workspace. Please upload or generate your dataset first.")

# Load the first detected dataset (or specify index/filename if needed)
dataset_path = csv_files[0]
print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)

# 2. Automatically select target and features
# (Replace 'target_label' with your actual target column name if different)
target_col = 'target_label' if 'target_label' in df.columns else df.columns[-1]
print(f"Using target column: '{target_col}'")

X = df.drop(columns=[target_col])
y = df[target_col]

# Retain only numeric features for modeling
X = X.select_dtypes(include=['number'])

# 3. Create honest split (Grouped by domain/client if present, else Stratified)
if 'client_id' in df.columns:
    groups = df['client_id']
    gkf = GroupKFold(n_splits=5)
    train_idx, val_idx = next(gkf.split(X, y, groups=groups))

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    print(f"✅ Grouped split created. Train shape: {X_train.shape}, Val shape: {X_val.shape}")
else:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() <= 10 else None
    )
    print(f"✅ Standard split created. Train shape: {X_train.shape}, Val shape: {X_val.shape}")

Found CSV files: ['./sample_data/mnist_test.csv', './sample_data/mnist_train_small.csv', './sample_data/california_housing_test.csv', './sample_data/california_housing_train.csv']
Loading dataset from: ./sample_data/mnist_test.csv
Using target column: '0.667'
✅ Standard split created. Train shape: (7999, 784), Val shape: (2000, 784)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Generate Week-4 Baseline Predictions on Validation Set
# (If your df already has a baseline column, use: y_pred_baseline = df.loc[X_val.index, 'baseline_pred'])
# As a standard heuristic baseline: predict 1 if traffic drop feature is above median, else 0
first_feature = X_val.columns[0]
y_pred_baseline = (X_val[first_feature] > X_val[first_feature].median()).astype(int)

# 2. Train the Week-5 Model (Random Forest)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Generate Baseline Predictions on Validation Set
first_feature = X_val.columns[0]
y_pred_baseline = (X_val[first_feature] > X_val[first_feature].median()).astype(int)

# 2. Train the Random Forest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Predict on Validation Set
y_pred_rf = model.predict(X_val)

# Safely extract probabilities for ROC-AUC
if hasattr(model, "predict_proba"):
    proba = model.predict_proba(X_val)
    # Check if the model predicted probabilities for both classes
    if proba.shape[1] > 1:
        y_proba_rf = proba[:, 1]
    else:
        y_proba_rf = None
else:
    y_proba_rf = None

# 4. Helper Function to Compute Evaluation Metrics
def evaluate_performance(y_true, y_pred, y_proba=None):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
    }
    # Compute ROC-AUC only if probabilities exist and true labels have more than 1 class
    if y_proba is not None and len(set(y_true)) > 1:
        metrics["ROC-AUC"] = roc_auc_score(y_true, y_proba)
    else:
        metrics["ROC-AUC"] = "N/A"
    return metrics

# 5. Build and Display Comparison Table
baseline_results = evaluate_performance(y_val, y_pred_baseline)
rf_results = evaluate_performance(y_val, y_pred_rf, y_proba_rf)

comparison_df = pd.DataFrame([baseline_results, rf_results], index=['Week 4 Baseline', 'Random Forest (Week 5)'])

print("=== Model vs. Baseline Comparison Table ===")
display(comparison_df.round(4))

=== Model vs. Baseline Comparison Table ===


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Week 4 Baseline,0.509,0.0,0.0,0.0,N/A
Random Forest (Week 5),1.000,0.0,0.0,0.0,N/A


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
import numpy as np
import pandas as pd

# 1. Feature Importance Analysis
if hasattr(model, "feature_importances_"):
    importances = pd.Series(model.feature_importances_, index=X_train.columns)
    importances = importances.sort_values(ascending=False)

    print("=== Top Feature Importances ===")
    display(importances.head(10))

# 2. Extract Misclassified Validation Samples (Error Analysis)
# Identify indices where prediction does not match actual target
errors_mask = y_val != y_pred_rf
error_indices = y_val[errors_mask].index

print(f"\nTotal Validation Errors: {len(error_indices)} out of {len(y_val)} rows")

if len(error_indices) > 0:
    # Build error inspection dataframe
    error_df = X_val.loc[error_indices].copy()
    error_df['Actual'] = y_val.loc[error_indices]
    error_df['Predicted'] = y_pred_rf[errors_mask]

    print("\n=== Sample Misclassified Rows (First 5) ===")
    display(error_df.head(5))
else:
    print("No validation errors detected.")


=== Top Feature Importances ===


,0
0.650,0.0
0.649,0.0
0.648,0.0
0.647,0.0
0.646,0.0
0.645,0.0
0.644,0.0
0.643,0.0
0.642,0.0
0.641,0.0



Total Validation Errors: 0 out of 2000 rows
No validation errors detected.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.